# Pipis 6.7 Ultra Hammamovich — QLoRA experiment

Экспериментальное дообучение Qwen2.5-1.5B-Instruct через QLoRA на Google Colab T4. Запускай сверху вниз. Это тестовый прототип, а не готовая модель.

In [ ]:
import torch, sys
print('Python:',sys.version.split()[0])
print('PyTorch:',torch.__version__)
print('CUDA:',torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('GPU не подключён. Выбери T4 GPU в настройках Colab.')
print('GPU:',torch.cuda.get_device_name(0))
print('VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))

## Установка

Важно: старая bitsandbytes 0.45.2 не содержит нужной CUDA 13.0 binary. Используем 0.50.2, где есть CUDA 13.x поддержка.

In [ ]:
%pip install -q transformers==4.48.3 datasets==3.2.0 peft==0.14.0 trl==0.14.0 accelerate==1.3.0 bitsandbytes==0.50.2 sentencepiece


In [ ]:
import sys,inspect,torch,transformers,datasets,peft,trl,accelerate,bitsandbytes
from trl import SFTTrainer,SFTConfig
print('Python:',sys.version.split()[0])
for p in (torch,transformers,datasets,peft,trl,accelerate,bitsandbytes): print(f'{p.__name__}: {p.__version__}')
print('Torch CUDA:',torch.version.cuda)
print('SFTTrainer processing_class:', 'processing_class' in inspect.signature(SFTTrainer.__init__).parameters)
sft_params=inspect.signature(SFTConfig.__init__).parameters
print('SFTConfig max_seq_length:', 'max_seq_length' in sft_params)
print('SFTConfig max_length:', 'max_length' in sft_params)
print('SFTConfig eval_strategy:', 'eval_strategy' in sft_params)
print('SFTConfig evaluation_strategy:', 'evaluation_strategy' in sft_params)
if not torch.cuda.is_available(): raise RuntimeError('CUDA недоступна.')
if not bitsandbytes.__version__.startswith('0.50'): raise RuntimeError(f'Нужен bitsandbytes 0.50.x, получен {bitsandbytes.__version__}.')
if 'processing_class' not in inspect.signature(SFTTrainer.__init__).parameters: raise RuntimeError('TRL не поддерживает processing_class.')
if not ({'max_seq_length','max_length'} & set(sft_params)): raise RuntimeError('TRL не поддерживает ограничение длины.')

In [ ]:
from datasets import load_dataset
train_url='https://raw.githubusercontent.com/shadowfightdimaonly-del/Pipis-6.7/main/data/pipis_train.jsonl'
eval_url='https://raw.githubusercontent.com/shadowfightdimaonly-del/Pipis-6.7/main/data/pipis_eval.jsonl'
train_dataset=load_dataset('json',data_files=train_url,split='train')
eval_dataset=load_dataset('json',data_files=eval_url,split='train')
print('Train rows:',len(train_dataset)); print('Eval rows:',len(eval_dataset))
def validate(ds,name):
    texts=[]
    for i,row in enumerate(ds):
        text=row.get('text')
        if not isinstance(text,str) or not text.strip(): raise ValueError(f'{name}[{i}] has invalid text')
        texts.append(text.strip())
    if len(texts)!=len(set(texts)): raise ValueError(f'{name} contains duplicate texts')
    return set(texts)
train_texts=validate(train_dataset,'train'); eval_texts=validate(eval_dataset,'eval')
overlap=train_texts & eval_texts
if overlap: raise ValueError(f'{len(overlap)} exact train/eval duplicates found')
print('Dataset validation passed.')

In [ ]:
import torch
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
model_name='Qwen/Qwen2.5-1.5B-Instruct'
tokenizer=AutoTokenizer.from_pretrained(model_name,use_fast=True)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
quant_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
model=AutoModelForCausalLM.from_pretrained(model_name,quantization_config=quant_config,device_map='auto',torch_dtype=torch.float16)
model.config.use_cache=False
print('Модель загружена в 4-битном режиме.')

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer,SFTConfig
lora_config=LoraConfig(r=8,lora_alpha=16,lora_dropout=0.05,bias='none',task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
sft_params=inspect.signature(SFTConfig.__init__).parameters
kwargs=dict(output_dir='/content/pipis-6.7-qlora-output',per_device_train_batch_size=1,per_device_eval_batch_size=1,gradient_accumulation_steps=4,learning_rate=2e-4,num_train_epochs=1,max_steps=10,logging_steps=1,eval_steps=5,save_strategy='no',report_to='none',fp16=True,gradient_checkpointing=True,optim='paged_adamw_8bit',dataset_text_field='text',packing=False)
kwargs['eval_strategy']='steps' if 'eval_strategy' in sft_params else 'steps' if 'evaluation_strategy' in sft_params else (_ for _ in ()).throw(RuntimeError('Нет eval strategy в SFTConfig'))
if 'evaluation_strategy' in sft_params and 'eval_strategy' not in sft_params: kwargs['evaluation_strategy']='steps'; kwargs.pop('eval_strategy',None)
kwargs['max_seq_length']=512 if 'max_seq_length' in sft_params else 512 if 'max_length' in sft_params else (_ for _ in ()).throw(RuntimeError('Нет параметра длины в SFTConfig'))
if 'max_seq_length' not in sft_params: kwargs.pop('max_seq_length',None); kwargs['max_length']=512
training_args=SFTConfig(**kwargs)
trainer=SFTTrainer(model=model,args=training_args,train_dataset=train_dataset,eval_dataset=eval_dataset,peft_config=lora_config,processing_class=tokenizer)
trainer.train()
eval_results=trainer.evaluate()
print('Eval results:',eval_results)

In [ ]:
adapter_dir='/content/pipis-6.7-adapter'
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print('Адаптер сохранён:',adapter_dir)

In [ ]:
import torch
from peft import PeftModel
reload_quant_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
reload_base=AutoModelForCausalLM.from_pretrained(model_name,quantization_config=reload_quant_config,device_map='auto',torch_dtype=torch.float16)
reloaded_model=PeftModel.from_pretrained(reload_base,adapter_dir); reloaded_model.eval()
prompt='Пользователь: Кто ты?\nПипис:'
inputs=tokenizer(prompt,return_tensors='pt').to(reloaded_model.device)
with torch.no_grad(): out=reloaded_model.generate(**inputs,max_new_tokens=80,do_sample=False)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],skip_special_tokens=True))

In [ ]:
def generate_answer(active_model,prompt,max_new_tokens=100):
    inputs=tokenizer(prompt,return_tensors='pt').to(active_model.device)
    with torch.no_grad(): out=active_model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],skip_special_tokens=True).strip()
prompts=['Пользователь: Кто ты?\nПипис:','Пользователь: Объясни LoRA простыми словами.\nПипис:','Пользователь: Что ты сделаешь, если не уверен в ответе?\nПипис:']
for prompt in prompts:
    print('\n===',prompt.split('\n')[0],'===')
    with trainer.model.disable_adapter(): base_answer=generate_answer(trainer.model,prompt)
    adapter_answer=generate_answer(trainer.model,prompt)
    print('BASE:',base_answer); print('ADAPTER:',adapter_answer)

In [ ]:
import shutil,os
zip_path='/content/pipis-6.7-adapter.zip'
shutil.make_archive('/content/pipis-6.7-adapter','zip',adapter_dir)
print('ZIP:',zip_path)
from google.colab import files
files.download(zip_path)